# Handling dino-loket Geotop x-section pdf files

This notebook contains a step by step handling of Geotop x-section pdf files.

This includes clicking to sample legend colors, extent and coordinates and the
the automatic sampling of the xsection on the image to get the legend index for
each geotop voxel.

Intermediate data are / have been pickled for later retrieval so that hand picking
of points on the images is by-passed.

After all relevant information was sampled, a dictionary of
x-section (Geotop_xsec class) objects is gerated and pickled as well.

These can be unpickled and yield all x-section information.

Not that some hand-work has been involved in collecting the relavant infromation

1. The world extent of each x-sections
2. The geo_unit ("geologische eenheid" en de "meest waarschijnlijke lithoclasse") which corresponds to the type of each geotop x-section

**Warning:**
Do not change add or remove the geotop pdf files from dirs.dino directory,
because all hand captured data correspond to the current files and their order.

To fill fdm grids with data from the cross section see

os.path.join(dirs.src, "ARK_fdm.py")
os.path.join(dirs.notebooks, "ARK_fdm.ipynb")
1. 

# Investigating the impact of the leakage from the Amsterdam Rijnkanaal

Low lying polders adjacent to the Amsterdam Rijnkanaal (ARK) experience more and more
leakage probably from the nearby ARK, which has a water level several meters higher
than that of the polders. The question is what causes this increase of the leagkage occurring
over the last two decades and how can it be solved by use of a layar of a sand-bentione mixture
at the canal bottom.

To investigate this, we'll generate cross section and model them in detail. With this or these
models, different impacts can be simulated and the impact possible measures can be examined for
their effectiveness.

The work is done for Rijkswaterstaat.

@ TO 2026-04-03

In [1]:
import os
import sys
from typing import Any
from glob import glob
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pdf2image
import pickle

from mf6lab.Projects.ARK_RWS.src.ARK_geotop import (
    parse_geotop_filename,
    Geotop_xsec,
    CrossSectionDigitizer,
    ImagePicker,
    Dirs,
    plot_result,    
    LITHO_CLASSES,
    GEO_UNITS
    )

print(sys.executable)

# --- Needed to make figure separate from the notebook and interactive
%matplotlib qt

# --- Seet the namespace for the relevant directories
dirs = Dirs()

# --- Get the paths and names of  the geotop pdf files in the order they are in dirs.dino
xsec_paths = {i:name for i, name in enumerate(glob(dirs.dino + '*.pdf'))}
xsec_names = {i:os.path.basename(name) for i, name in enumerate(glob(dirs.dino + '*.pdf'))}

# --- Pickling
def pickleto(var:Any, basename:str, parent:str=dirs.data):
    """Pickle var to os.path.join(dirs.data, basename)"""
    if not basename.endswith('.pkl'):
        basename += ".pkl"

    pkl_file = os.path.join(parent, basename)
    with open(pkl_file, 'wb') as f:
        print(f"Pickled {basename} --> {parent}")        
        pickle.dump(var, f)

# --- Unpickling
def picklefrom(basename:str, parent:str=dirs.data)->Any:
    """Unpickle varname from os.path.join(parent, basename)"""
    if not basename.endswith(".pkl"):
        basename += ".pkl"
          
    pkl_file = os.path.join(parent, basename)
    with open(pkl_file, 'rb') as f:
        print(f"Loaded {basename} <-- {parent}")        
        return pickle.load(f)

# --- Color for empty legend (empty voxel with geo_unit 'none')
WHITE_01 = np.array([1., 1., 1.])

loading mfpath.py
/Users/Theo/Development/python/mf6_tools/mf6lab/.venv/bin/python


# Get the geotop.pdf obtained from dinoloket.nl

When reading it with pdf2image you get actually two pdfs.
One has the actual cross section and the other the legend and a map.

In [ ]:
# --- Show images one after the other

for i, geotop_pdf in xsec_paths.items():
    basename = os.path.basename(geotop_pdf)
        
    # --- The dpi=300 determines the pixel size of the image !
    # --- Results in one PIL object for each page in the pdf file.
    geotop1, geotop2 = pdf2image.convert_from_path(geotop_pdf, dpi=300)
    
    # --- Convert to RGB (each color as int 0-255)
    geotop1 = np.asarray(geotop1.convert("RGB"))  
    geotop2 = np.asarray(geotop2.convert("RGB"))
    
    # --- Imshow uses 0-255 as color if dtype is ints and 0-1. when dtype is float
    # --- Just to show it, doesn't matter.
    
    # --- The first pdf page contains the x-section
    fig, ax = plt.subplots(figsize=(10, 6))
    fig.suptitle(basename)
    ax.set_title("Geotop Cross Section")
    ax.imshow(geotop1) 
    ax.plot()
    
    # --- The second pdf page contains the legend and a small map.
    fig, ax = plt.subplots(figsize=(10, 6))
    fig.suptitle(basename)
    ax.set_title("Geotop, legend and map")
    ax.imshow(geotop2)
    ax.plot()
    

# Example (using the last geotop pdf file in dirs.dino)

Get the last pdf file in the dirs.dino directory.
Show its properties.

## Get the file its, contents and show its properties

In [ ]:
print("Specifications of the Geotop pdf")
print("================================")
print("Directory dirs.dino:")
print(dirs.dino)
print()
# --- Get the last geotop.pdf file in dirs.dino
geotop_pdf = glob(dirs.dino + '*.pdf')[-1]

print("Filename:")
print(os.path.basename(geotop_pdf))
print()

print("File contents:")
print("==============")
# --- Convert the pdf to image. Each page in the pdf becomes a PIL object
# --- Note that the dpi argument determines the pixel size of the image
geotop_pages = pdf2image.convert_from_path(geotop_pdf, dpi=300)
print(f"Number of page in geotop pdf file: {len(geotop_pages)}")

# --- Get the two pages
geotop1, geotop2 = geotop_pages

# --- Show their types
print(f"Type geotop1: {type(geotop1)}")
print(f"Type geotop2: {type(geotop2)}")

# --- Convert the PIL ismage to RGB
geotop1 = np.asarray(geotop1.convert("RGB"))  
geotop2 = np.asarray(geotop2.convert("RGB"))

# --- Show their shape
print(f"geotop1 shape = {geotop1.shape}")
print(f"geotop2.shape = {geotop2.shape}")

# --- Show their RGB dtype and range
print(f"geotop1.dtype = {geotop1.dtype}, RGB values: {geotop1.min()}-{geotop1.max()}")
print(f"geotop2.dtype = {geotop2.dtype}, RGB values: {geotop2.min()}-{geotop2.max()}")


## Hand-written geoCodes and world-extent for this cross section

We will combine the legend colors with the hand written geoCodes pertaining to this
cross section. They are paired.

And later on with the hand-written world_extent of this cross section. The vertical
z is in m relatieve to NAP (National Datum) while the x-coordinate is in m relative
to the left-most point of the cross section. Notice that the actual national
coordinates of this starting point is in the file name and that a small
map showing where the cross section is, is in the second page of the geotop's pdf file.

In [ ]:
geoCodes = ['NUECga', 'NUECgb', 'NUEC1', 'NUNIHO', 'NUNIBA', 'NUBXWI-SI-KO', 'NUBX', 'NUDR', 'NUgs']

# --- extent is always (xmin, xmax, zmin zmax). In this case in m and z (vertical)
world_extent=(0, 8625, -48.5, 0)

## Use the image picker with the legend image to get the legend colors

Instantiate the ImagePicker with the image that holds the map with the legend color boxes.
Then zoom in and click the legend's color boxes in the normal order.
This yields the colors to compare those sampled in the actual cross section with.

Press ENTER to finish with the legend.

In [ ]:
# ---Instantiate the picker with the image to pick from (image with the legend)
picker = ImagePicker(geotop2)

# --- Zoom before clicking (zoom into legend). The easiest way is the shift the figure
# --- to hit the top panel of the screen. The shifted window will go full screen automatically.

# --- Step 1: Pick legend colors one after the other in sequence. Press ENTER when done.
colors = picker.get_colors(n=-1)

# --- Special for legend colors Remove white points first to remove mistake clicks
legend_colors = np.array([clr for clr in colors if not np.all(clr > 0.95)])

# --- Internally set the legend_colors (obtained from clicking the legend)
legend_colors = np.vstack((np.array([WHITE]), legend_colors))
geoCodes.insert(0, 'none')


# --- Show the legend_color geoCode pairs for this cross section.
print("Legend_color     geoCode")
print("============     =======")
for l_color, g_code in zip(legend_colors, geoCodes):
    print(np.round(l_color, 2), g_code)


## With the legend_colors now obtained get the pixel extent of the cross section

To get the pixel extent of the cross section:

1. Instantiate the ImagePicker once again, but now with the image of the actual cross section.
2. Zoom in the same way has before. Preferably by shifting the image window to the top of the screen to let it go full screen.
3. Then pick the corners of the cross sections of which you know the world coordinates.
4. Press ENTER when finished. The pixel_exent is thus obtained.

In fact, a pixel bounding box is computed around all picked points and turned into the
pixel_extent: (pxmin, pxmax, pymin, pymax)


In [ ]:
# --- Initiate a new picker, now with the cross section   
# --- To get the pixel bounding box
picker = ImagePicker(geotop1)

# --- Again, zoom in
# --- In fact you can click any number of points, the bbox uses there min and max coords
pxl_extent = picker.get_pxl_bbox(n=-1)

print("Pixel extent of the cross section:")
print("==================================")
print("[pxmin pxmax pymin pymax]")
print(np.array(pxl_extent))

## Sample the actual cross section automatically

Sampling the actual cross section image for to match each
sampled point with the legend is done automatically based on

1. The legend colors.
2. The pixel extent.
3. The world extent.
4. The voxel size if the geotop image (dx, dz), generally (100, 0.5)

The subdivision of the grid is based on the world_extent and the voxel size dx, dz.

To match empyt voxels:
1. 'none' is prepended to the geoCodes
2. WHITE = np.array([1., 1., 1.]) is prepenced to the legend_colors

The voxel array will be filled with legend codes.
Code 0 indicates empty (no legend) and get legend 'none' if required.

In [ ]:
# --- Fill an array with soil indices where each index is the number of the
# --- legend color boxes in that order (the order clicked before)

# --- Instantiate the CrossSectionDigitizer using the cross section.
digitizer = CrossSectionDigitizer(image=geotop1)

# --- Set pxl_extent. We just obtained it
digitizer.set_pxl_bbox(*pxl_extent)

# --- Set world extent (world_extent was given above)
digitizer.set_world_bbox(*world_extent)

digitizer.legend_colors = legend_colors

# --- Set voxel size.
digitizer.set_grid(dx=100, dz=0.5)

# --- Fill an array of a cross section shape obtaine from dx, dz and world_extent
# --- This is done inside build
arr = digitizer.build_array()

## Show he legend indices of the sampled the cross section

The digitizer's build_array contains the legend indices for each voxel.
Empty voxels have index 0. The largest index is the index of the last legend
index (but with 'none' as the first).

The sampling was done af follows:
1. The color values of a small patch around the centre of each voxel are used.
2. Dark colors are removed to get rid of text (using brightness computation).
3. The median of the colors is taken.
4. The distance in RGB from this color to the legend_colors is computed
5. The nearest legend_color's index is found and filled into the arr array.

The resulting voxel lhrnf index array can subsequently be converted to
any property that can be linked to the legend. The best way to do this is by
using a pandas DataFrame or a dictionary with the same index as the legend colorboxes.

The underlying grid is that defined by the world_extent and the (dx,dz) voxel size.
So arr.shape == geoImage.shape(:2). The latter has RGB values and is 3D.

An mfgrid.Grid object can be used as an alternative grid, but this is not
recommended because it can create artefacts: When the resolution is too fine
these artefacts are caused by lines and text in the image.

Resampling on other grids should be done separately.

In [ ]:
# --- Show the cross section using imshow, which fills the voxels
plot_result(arr, world_extent)

# --- Add title and save
fig = plt.gcf()
fig.suptitle(f"""{os.path.basename(geotop_pdf)}
                with colors converted to soil-indices
                """)

# --- Save the image for later reporting
fig.savefig(os.path.join(dirs.images, f"{os.path.basename(geotop_pdf)}"))

plt.show()

This finishes the work flow. But with a 

# Prepare cross section data for all geotop pdf files at once (in a loop)

## 1. Get the xRD, yRD coordinates of the cross sections

In [ ]:
# --- Coordinate data for the various cross sections

# --- Dictionary to store the picked points (pixels)
geotopMapPoints = {}

# --- Run over all the geotop files stored.
for _, geotop_pdf in xsec_paths.items():
    
    # --- Use  basename as key.
    basename = os.path.basename(geotop_pdf)
    
    # --- Get the two pages of each geotop pdf as RGB image 
    geotop_xsec, geotop_leg = pdf2image.convert_from_path(geotop_pdf, dpi=300)
        
    geotop_xsec = np.asarray(geotop_xsec.convert("RGB"))  # map
    geotop_leg  = np.asarray(geotop_leg.convert("RGB"))  # legend (not used here) 
    
    # --- Instantiate the point picker with the map image
    picker = ImagePicker(geotop_leg)
    
    # --- Click 2 map bbox points of know coordinates
    # --- followed determining the cross sections on the map
    points = picker.pick_points(n=-1, title=basename)

    # --- Store them
    geotopMapPoints[basename] = {'pts': np.array(points)}

Pickle the geotopMapPoints

In [ ]:
pickleto(geotopMapPoints, 'geotopMapPoints.pkl')

## 2. Get the legend of the geotop xsections

### 2.1 Get the legend colors of each geotop X-section

The legend is the second page of each geotop pdf file. Each
image is loaded in turn and the colors of the legend are clicked
and stored in a dictionary whose keyse are the basename of the
mentioned files.

These colors should be matched with the labels of the legends.
But these labels have to be provided by the user and are
given below for the current geotop pdfs


In [ ]:
# --- Dictionary to store the legend_colors obained by the picked points
geotop_leg_colors = {}

# --- Run over all the geotop files stored.
for _, geotop_pdf in xsec_paths.items():
    
    # --- Use  basename as key.
    basename = os.path.basename(geotop_pdf)
    
    # --- Get the two pages of each geotop pdf as RGB image 
    geotop_xsec, geotop_leg = pdf2image.convert_from_path(geotop_pdf, dpi=300)
        
    geotop_xsec = np.asarray(geotop_xsec.convert("RGB"))  # map
    geotop_leg  = np.asarray(geotop_leg.convert("RGB"))  # legend (not used here) 
    
    # --- Instantiate the point picker with the map image
    picker = ImagePicker(geotop_leg)
    
    # --- Click the 5 points
    leg_colors = picker.get_colors(n=-1)

    # --- Add the legend colors of this xsec in the dict
    geotop_leg_colors[basename] = leg_colors

### 2.2 Get the legend labels of the available cross sections

The legend labels are listed below in the order of the geotop pdf files in the
directory dirs.dino. The order of the files and labels are crucial. The
labels will change when the geotop pdf files are added or deleted or their names changed.

In [ ]:
leg_labels = [
    "NUAAOP	NUECgb	NUEC1	NUNIHO	NUNIBA	NUNBXWI-SI-KO	NUBX	NUKR-BXDE	NUDR	NUgs	NUUR2	NUST",
    "a	v	k	kz	zf	zm	zg	g	she",
    "a	v	k	kz	zf	zm	zg	g	she",		
    "NUECga	NUECgb	NUEC1	NUNIHO	NUNIBA	NUNBXWI-SI-KO	NUBX	NUKR-BXDE	NUDR	NUgs	NUUR2	NUST",
    "NUAAOP	NUECga	NUECgb	NUEC1	NUNIHO	NUNIBA	NUNBXWI-SI-KO	NUBX	NUDR	NUgs	NUST",
    "NUAAOP	NUECga	NUECgb	NUEC1	NUNIHO	NUNIBA	NUNBXWI-SI-KO	NUBX	NUKR-BXDE	NUDR	NUgs	NUUR2	NUST",
    "NUAAOP	NUECgb	NUEC1	NUNIHO	NUNIBA	NUNBXWI-SI-KO	NUBX	NUKR-BXDE	NUDR	NUgs	NUUR2	NUST",
    "a	v	k	kz	zf	zm	zg	g	she",
    "a	v	k	kz	zf	zm	zg	g	she",
    "a	v	k	kz	zf	zm	zg	g	she"
]

geolegs = {}

for (fname, colors), label in zip(geotop_leg_colors.items(), leg_labels):    

    # --- We must use white as a legend option for empty cells otherwise
    # --- we can never correctly compute the distance in RGB space
    # --- between empty points and the other legends.
    # --- Therefore we also need a 'none' geo_unit as well.
    
    # --- Put the empty cell in front
    clrs = list(colors)
    lbls = label.split('\t')
    
    clrs.insert(0, WHITE_01)
    lbls.insert(0, 'none')

    # --- Store the labels and colors of the legend    
    geolegs[fname] = {'leg_labels': lbls, 'leg_colors':clrs}

geolegs

#### Pickle the geolegs dictionary

In [ ]:
pickleto(geolegs,  'geotop_legends.pkl')

#### Unpickle the geotop legends

In [ ]:
geolegs = picklefrom('geotop_legends.pkl')

# 3. Georeferencing the X-sections

## 3.1 Get the point data necessary to georeference the different cross sections

Get the getop XZ points to vertically reference the cross section.

we loop through the getop x-sections. Within each of them we click 5 points:
    
    3 on the vertical z-axis and
    2 on the horizontal x-axis.

The first 3 points are at the vertical line where x=0:

    1. The point z=0
    2. The point z = the lowest z-tick at x=0
    3. The bottom point of the x-section at x=0
   
The 2 points on the x-axis are all at the bottom of the x-section:

    1. At the last (right-most) x-tic
    2. At the right-most point of the x-section

This provides the points form which the x-section can be vertically referenced.


In [ ]:
# --- Coordinate data for the various cross sections

# --- Dictionary to store the picked points (pixels)
geoXZ_pts = {}

# --- Run over all the geotop files stored.
for _, geotop_pdf in xsec_paths.items():
    
    # --- Use  basename as key.
    basename = os.path.basename(geotop_pdf)
    
    # --- Get the two pages of each geotop pdf as RGB image 
    geotop_xsec, geotop_leg = pdf2image.convert_from_path(geotop_pdf, dpi=300)
        
    geotop_xsec = np.asarray(geotop_xsec.convert("RGB"))  # map
    geotop_leg  = np.asarray(geotop_leg.convert("RGB"))  # legend (not used here) 
    
    # --- Instantiate the point picker with the map image
    picker = ImagePicker(geotop_xsec)
    
    # --- Click the 5 points
    points = picker.pick_points(n=-1, title=basename)

    # --- Store them
    geoXZ_pts[basename] = points
    

## 3.2 Convert the points --> dict

The 3 points along the z-axis are to determine the exact vertical z-extent
The third point is bottom left point of the xsec (its origin)
The 2 points along the x-axis determing the x-extent of the xsec.

The 5 sampled coordinate pairs --> into coordinate values:
xp1, zp1, xp2, zp2, xp3, zp3, xp4, zp4, xp5, zp5

They are first converted into a dictionary.
This dictionary is then converted into a pd.DataFrame with columns
naming the points.

In [ ]:
# === More convenient dict ====
db = geoXZ_pts.copy() # --- Don't destroy original

# --- Convert the collected coordinate pairs into the z and x coordinates
for k in db.keys():
    
    # --- first the 5 coordinate pairs of each geotop X-section --> array
    pts = np.array(db[k])
    
    # --- Select the 3 z-values and the 3 x-values
    db[k] = {'px1': pts[0, 0], 'pz1': pts[0, 1],
             'px2': pts[1, 0], 'pz2': pts[1, 1],
             'px3': pts[2, 0], 'pz3': pts[2, 1],
             'px4': pts[3, 0], 'pz4': pts[3, 1],
             'px5': pts[4, 0], 'pz5': pts[4, 1]}


# ====  Convert to pd.DataFrame ===
# --- Generate a DataFrame to hold the extents of all the geotop_pdf files
df_geoXZ_pts = pd.DataFrame(index=db.keys(),
                          columns=db[k].keys()
                          )

# --- Fill it with the captured points
for k in db.keys():   
    df_geoXZ_pts.loc[k] = db[k]

# --- Make sure their type is not object, but int.
for col in df_geoXZ_pts.columns:
    df_geoXZ_pts[col] = df_geoXZ_pts[col].astype(int)
    
# --- Show the DataFrame
df_geoXZ_pts

## 3.3 Add corresponding real-world values

The corresponding real-world value of Pz1 (z1) and Pz2 (z2) as well as Px1 (x1) and Px2 (x2) have been visually picked from the geoXsec images. They are shown below and
are converted into a pd.DataFrame.

When done, these two DataFrames are merged and the world_extent values z3 and x3 are
computed and then added to the DataFrame.

In [ ]:
# --- Coordinates z1, z2, x1, x2 of the geotop cross sections
df_wrld_axes = pd.DataFrame(data=np.array([
                      [0.0, -45.0, -99.0, 0.0, 8400.0, 9999.0],
                      [0.0, -45.0, -99.0, 0.0, 5600.0, 9999.0],
                      [0.0, -45.0, -99.0, 0.0, 8250.0, 9999.0],
                      [0.0, -48.0, -99.0, 0.0, 7000.0, 9999.0],
                      [0.0, -45.0, -99.0, 0.0, 4800.0, 9999.0],
                      [0.0, -45.0, -99.0, 0.0, 8250.0, 9999.0],
                      [0.0, -45.0, -99.0, 0.0, 5600.0, 9999.0],
                      [0.0, -48.0, -99.0, 0.0, 7000.0, 9999.0],
                      [0.0, -45.0, -99.0, 0.0, 4800.0, 9999.0],
                      [0.0, -45.0, -99.0, 0.0, 8400.0, 9999.0]
                      ]),
             index=db.keys(), # geotop pdfs basenames
             columns = ['z1', 'z2', 'z3', 'x1', 'x2', 'x3']
)

Compute and add the zmin and xmax in world coordinates

In [ ]:
# --- Compute the bottom coordinate of the z-axis
dfe = df_wrld_axes
dfe['z3'] = (dfe['z1'] + 
              (dfe['pz3'] - dfe['pz1']) / (dfe['pz2'] - dfe['pz1']) *
              (dfe['z2']  - dfe['z1'])
)

# --- Compute the largest coordinate of the x-axis
dfe['x3'] = (dfe['x1'] + 
              (dfe['px5'] - dfe['px3']) / (dfe['px4'] - dfe['px3']) *
              (dfe['x2']  - dfe['x1'])
)

# --- Just round for convenience
dfe['x3'] = np.round(dfe['x3'])

# --- Show the extents DataFrame, which is now complete
df_wrld_axes

## 3.4 Merge the two data frames

In [ ]:
# --- Merge the two DataFrames
for col in df_wrld_axes.columns:
    df_geoXZ_pts[col] = df_wrld_axes[col]

# --- Show the extents DataFrame
df_geoXZ_pts

In [ ]:
# --- Reduce the columns to just contain the pxl and world extent values
df_extents = df_geoXZ_pts[['px1', 'px5', 'pz1', 'pz5', 'x1', 'x3', 'z3', 'z1']]
df_extents.columns = ['pxmin', 'pxmax', 'pymin', 'pymax', 'xmin', 'xmax', 'zmin', 'zmax']

pickleto(df_extents, 'geoXZ_extents.pkl')

Show the world-extents and pxl_extents for all the geoptop pdf files

In [ ]:
print("World extent of the subsequent X-sections")
print("=========================================")
print("xmin   xmax   zmin   zmax")
for fn in df_geoXZ_pts.index:
    rec = df_geoXZ_pts.loc[fn]
    print(rec['x1'], rec['x3'], rec['z1'], rec['z3'])
    
print()

print("Pixel extent of the subsequent X-sections")
print("=========================================")
print("pxmin  pxmax  pymin  pymax")
for fn in df_geoXZ_pts.index:
    rec = df_geoXZ_pts.loc[fn]
    print(int(rec['px3']), int(rec['px5']), int(rec['pz1']), int(rec['pz3']))

Pickle df_geoXZ_pts

In [ ]:
pickleto(df_geoXZ_pts, 'geotop_points.pkl')

# 4. Generating Geotop_xsec class dictionary

# 4.1 Unpickle the already stored data sources

Instead of going through the pipepline process of picking the legend colors
and the pxl-extents, we start here by retrieving the data from disk directly.

In [2]:
geolegs = picklefrom('geotop_legends.pkl')

df_geoXZ_pts = picklefrom('geotop_points.pkl')

df_extents = picklefrom('geoXZ_extents.pkl')
    
geotopMapPoints = picklefrom('geotopMapPoints.pkl')

Loaded geotop_legends.pkl <-- /Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/data/
Loaded geotop_points.pkl <-- /Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/data/
Loaded geoXZ_extents.pkl <-- /Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/data/
Loaded geotopMapPoints.pkl <-- /Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/data/


## 4.2 Generated the geotop_xsecs dictionary

Using the CrossSectionDigitizer class, automatically sample the cross section
colors and match these with the legend colors to fill an array with the
legend index of which zero means none and the legend index actually start with 1.

1. The legend colors are in geolegs[fname]['colors']
2. The legend units are in geoloegs[fname]['labels']

Gather all relevant information and generate the xsecs dict.

In [3]:
geotop_xsecs = {}

# --- Run over all the geotop files stored.
for _, geotop_pdf in xsec_paths.items():
    fname = os.path.basename(geotop_pdf)
    
    # --- Parse filename toextract xsec_type, xRD and yDR
    xsec_type, xRD, yRD = parse_geotop_filename(fname)
    
    # -- Get the extents of the current X-section
    rec = df_extents.loc[fname]
    pxl_extent   = (rec['pxmin'], rec['pxmax'], rec['pymin'], rec['pymax'])
    world_extent = (rec['xmin'],  rec['xmax'],  rec['zmin'],  rec['zmax'])
    
    # --- Get the pixel coordinates of xsec on the small map in the geotop pdf p2
    xy_map_pxl = geotopMapPoints[fname]['pts']

    # --- Get the legend color and label of the current X-section
    legend_colors = geolegs[fname]['leg_colors']
    legend_labels = geolegs[fname]['leg_labels']
        
    # --- Get the image of the X-section of the current geotop pdf
    xsec_image = pdf2image.convert_from_path(geotop_pdf, dpi=300)[0]
    xsec_image = np.asarray(xsec_image.convert("RGB"))  
    
    # --- Instantiate the digitizer
    digitizer = CrossSectionDigitizer(xsec_image)
    
    # --- Fill the parameters of the digitizer to do its work
    digitizer.legend_colors = legend_colors
    digitizer.set_pxl_bbox(*pxl_extent)
    digitizer.set_world_bbox(*world_extent)
    digitizer.set_grid(dx=100., dz=0.5)   # Could rather be stored before.
    
    # --- Let the digitizer fill an array with legend indices
    leg_index_array = digitizer.build_idx_array()
    
    geo_units = GEO_UNITS if 'geolog' in xsec_type else LITHO_CLASSES

    # --- Store all X-sec info in a single dictionary of Geotop_xsec objects    
    geotop_xsecs[fname] = Geotop_xsec(
                                fname=fname,
                                xsec_type=xsec_type,
                                xRD=xRD,
                                yRD=yRD,
                                pxl_extent=pxl_extent,
                                world_extent=world_extent,
                                xy_map_pxl=xy_map_pxl,
                                leg_colors=geolegs[fname]['leg_colors'],
                                leg_labels=geolegs[fname]['leg_labels'],     
                                geo_units=geo_units,
                                idx_arr=leg_index_array
    )
    
pickleto(geotop_xsecs, 'geotop_xsecs.pkl')

Pickled geotop_xsecs.pkl --> /Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/data/


Show the cross sections with leg_idx colored using colormap 'viridis'

In [5]:
for name, xsec in geotop_xsecs.items():
    # --- Incides as cmap 'viridis'
    xsec.plot_array(arr=None, par_name=None)
plt.show()

Show the cross section in their origional colors

In [ ]:
for name, xsec in geotop_xsecs.items():
    # --- With original legend colors
    xsec.show_leg_index_array()
    
plt.show()

# Get the property arrays for each X-section

Each X-section has its index array, world_extent and Geo_units with
property values per unit. From this a Voxel array can be fille with propery values.

During the loop we simply add them to the objects.


In [5]:
for name, xsec in geotop_xsecs.items():
    # --- With original legend colors
    xsec.props = xsec.get_props()
    
pickleto(geotop_xsecs, 'geotop_xsecs_with_props.pkl')


Pickled geotop_xsecs_with_props.pkl --> /Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/data/


In [6]:
for name, xsec in geotop_xsecs.items():
    print(name)
    print(xsec.props)

BRO GeoTOP Verticale doorsnede geologische eenheid 130173,479431.pdf
{'kh': array([[ 0.,  0.,  0., ..., 10.,  0.,  0.],
       [ 0.,  0.,  0., ...,  5.,  0.,  0.],
       [ 0.,  0.,  0., ...,  2.,  0.,  0.],
       ...,
       [40., 40., 40., ..., 40., 40., 40.],
       [40., 40., 40., ..., 40., 40., 40.],
       [40., 40., 40., ..., 40., 40., 40.]], shape=(100, 86)), 'kv': array([[0. , 0. , 0. , ..., 1. , 0. , 0. ],
       [0. , 0. , 0. , ..., 5. , 0. , 0. ],
       [0. , 0. , 0. , ..., 0.2, 0. , 0. ],
       ...,
       [4. , 4. , 4. , ..., 4. , 4. , 4. ],
       [4. , 4. , 4. , ..., 4. , 4. , 4. ],
       [4. , 4. , 4. , ..., 4. , 4. , 4. ]], shape=(100, 86)), 'n': array([[0.  , 0.  , 0.  , ..., 0.35, 0.  , 0.  ],
       [0.  , 0.  , 0.  , ..., 0.4 , 0.  , 0.  ],
       [0.  , 0.  , 0.  , ..., 0.38, 0.  , 0.  ],
       ...,
       [0.35, 0.35, 0.35, ..., 0.35, 0.35, 0.35],
       [0.35, 0.35, 0.35, ..., 0.35, 0.35, 0.35],
       [0.35, 0.35, 0.35, ..., 0.35, 0.35, 0.35]], shape=(100

### Fill gaps in the idx_arr with nearest horizontal neighbour

The trick is to overwrite idx_arr in each cross section. But not to destroy the
current geotop_xsecs, first copy the geotop_xsecs into a new dictionary
and replace the idx_arr in them with the filled one

In [7]:
geotop_xsecs_filled = picklefrom('geotop_xsecs.pkl')

for name, xsec in geotop_xsecs_filled.items():
    print(name)
    # --- Compute the filled array
    idx_arr = xsec.fill_horizontal(skip_valids=15, arr=None)
    
    # --- Replace the idx_arr in the xsec with the filled one
    xsec.idx_arr = idx_arr
    
    # --- Show the filled cross section
    xsec.show_leg_index_array()    
plt.show()

pickleto(geotop_xsecs_filled, 'geotop_xsecs_filled')

Loaded geotop_xsecs.pkl <-- /Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/data/
BRO GeoTOP Verticale doorsnede geologische eenheid 130173,479431.pdf
Plotting: BRO GeoTOP Verticale doorsnede geologische eenheid 130173,479431.pdf
BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 126311,474187.pdf
Plotting: BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 126311,474187.pdf
BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 130049,479466.pdf
Plotting: BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 130049,479466.pdf
BRO GeoTOP Verticale doorsnede geologische eenheid 129926,479462.pdf
Plotting: BRO GeoTOP Verticale doorsnede geologische eenheid 129926,479462.pdf
BRO GeoTOP Verticale doorsnede geologische eenheid 127484,477893.pdf
Plotting: BRO GeoTOP Verticale doorsnede geologische eenheid 127484,477893.pdf
BRO GeoTOP Verticale doorsnede geologische eenheid 130049,479466.pdf
Plotting: BRO GeoTOP Verticale doors